# Pipeline Preprocessing, Cleaning, & Agregasi Data Pengeluaran & Kemiskinan BPS (2018-2024)

Notebook ini memproses **14 file Excel** (11 kelompok komoditas dengan subkomoditas + 3 indikator kemiskinan/ketahanan pangan tingkat Kabupaten/Kota) dari BPS:
1. Ekstraksi otomatis metadata hierarki Komoditas Utama (KAPITAL) dan Subkomoditas (lowercase) untuk filter dinamis webstory/dashboard.
2. Pembersihan *messy header*, handling nilai strip (`'-'`) dan missing value.
3. Agregasi horizontal dinamis (`sum`) untuk tahun berulang antar subkomoditas.
4. Penggabungan berurutan (*outer merge*) pada seluruh 14 dataset.
5. Standardisasi dan filter data tepat **511 Kabupaten/Kota** (sejajar dengan peta spasial GeoJSON).
6. Export master dataset ke file CSV.

### Cell 1: Setup & Import Library
Mengimpor library yang diperlukan untuk manipulasi data tabular, ekstraksi JSON, dan operasi fungsional.

In [ ]:
import os
import re
import glob
import json
import numpy as np
import pandas as pd
from functools import reduce

# Path direktori dataset raw pengeluaran
DATA_DIR = r"dataset raw/pengeluaran"
TARGET_YEARS = [str(y) for y in range(2018, 2025)]  # Range: 2018 s.d. 2024

print(f"Setup berhasil. Target tahun pengamatan: {TARGET_YEARS}")

### Cell 2: Ekstraksi Metadata & Karakteristik Data (Komoditas Utama & Subkomoditas)
Mengekstrak hierarki kategori komoditas utama (UPPERCASE) dan daftar subkomoditas (lowercase) dari baris metadata file Excel komoditas.

In [ ]:
def extract_metadata(file_path):
    """
    Mengekstrak metadata hierarki dari 3 baris teratas file komoditas Excel.
    - Nama Komoditas Utama: teks non-null pertama pada Baris 2 (Index 1) -> UPPERCASE.
    - Daftar Subkomoditas: seluruh nilai unik berikutnya pada Baris 2 (Index 1) -> lowercase.
    """
    # Baca 3 baris pertama tanpa menganggapnya sebagai header
    df_peek = pd.read_excel(file_path, header=None, nrows=3)
    
    # Baris 2 (Index 1) memuat nama kategori komoditas utama dan sub-komoditas
    row_items = [str(x).strip() for x in df_peek.iloc[1].dropna().tolist() if str(x).strip()]
    
    if not row_items:
        return None, []
        
    main_commodity = row_items[0].upper()
    sub_commodities = list(dict.fromkeys([item.lower() for item in row_items[1:]]))
    
    return main_commodity, sub_commodities

# Daftar 11 file kelompok komoditas pengeluaran
komoditas_files = [
    "bahan makanan lainnya.xlsx",
    "buah.xlsx",
    "daging.xlsx",
    "ikan.xlsx",
    "kacang.xlsx",
    "makanan jadi.xlsx",
    "padi.xlsx",
    "rokok.xlsx",
    "sayur.xlsx",
    "telur susu.xlsx",
    "umbi.xlsx"
]

metadata_komoditas = {}
print("=" * 75)
print("HASIL EKSTRAKSI METADATA HIERARKI KOMODITAS & SUBKOMODITAS BPS")
print("=" * 75)

for fname in komoditas_files:
    fpath = os.path.join(DATA_DIR, fname)
    if os.path.exists(fpath):
        main_com, subs = extract_metadata(fpath)
        metadata_komoditas[main_com] = subs
        print(f"\n[KOMODITAS UTAMA] : {main_com}")
        print(f"  Jumlah Subkomoditas : {len(subs)}")
        print(f"  Daftar Subkomoditas : {', '.join(subs)}")

# Simpan metadata ke file JSON untuk integrasi webstory/dashboard interaktif
with open("metadata_subkomoditas.json", "w", encoding="utf-8") as f_meta:
    json.dump(metadata_komoditas, f_meta, indent=2, ensure_ascii=False)
print("\n[V] Metadata berhasil disimpan ke file 'metadata_subkomoditas.json'")

### Cell 3: Fungsi Cleaning & Agregasi Otomatis (`process_excel`)
Fungsi modular ini mampu menangani perbedaan struktur:
- **`is_komoditas=True` (11 file)**: Mengambil header di Baris 3 (Index 2). Memiliki subkomoditas sehingga kolom tahun berulang, lalu diagregasikan dengan horizontal sum (`.T.groupby(level=0, sort=False).sum().T`).
- **`is_komoditas=False` (3 file indikator)**: Mengambil header di Baris 2 (Index 1). Kolom tahun sudah unik sehingga proses sum dilewati.

In [ ]:
def process_excel(file_path, is_komoditas=True):
    """
    Memproses, membersihkan, dan mengagregasi data Excel BPS:
    1. Load data sesuai posisi header baris tahun.
    2. Filter kolom tahun '2018' sampai '2024'.
    3. Cleaning nilai strip ('-') dan spasi menjadi 0.0 (float).
    4. Agregasi dinamis (sum horizontal) jika kolom tahun berulang.
    5. Rename kolom dengan prefix nama file.
    """
    base_name = os.path.splitext(os.path.basename(file_path))[0].strip()
    header_row = 2 if is_komoditas else 1
    
    # 1. Load Data
    df = pd.read_excel(file_path, header=header_row)
    region_col_name = df.columns[0]
    
    # 2. Filter Tahun (2018 - 2024) & pertahankan Kolom Wilayah
    valid_col_indices = [0]  # Kolom A (Kabupaten/Kota)
    year_labels = []
    
    for idx, col in enumerate(df.columns[1:], start=1):
        # Bersihkan suffix duplicate bawaan pandas (misal '2018.1' -> '2018')
        year_str = str(col).split('.')[0].strip()
        if year_str in TARGET_YEARS:
            valid_col_indices.append(idx)
            year_labels.append(year_str)
            
    filtered_df = df.iloc[:, valid_col_indices].copy()
    filtered_df.rename(columns={region_col_name: "Kabupaten/Kota"}, inplace=True)
    
    # Cleaning kolom nama wilayah
    filtered_df["Kabupaten/Kota"] = filtered_df["Kabupaten/Kota"].astype(str).str.strip()
    
    # Standardisasi penamaan wilayah khusus (misal variasi huruf kecil 'dan')
    filtered_df["Kabupaten/Kota"] = filtered_df["Kabupaten/Kota"].replace({
        "Pangkajene Dan Kepulauan": "Pangkajene dan Kepulauan"
    })
    
    # Hapus baris non-kabupaten: catatan kaki, NaN, agregat provinsi (huruf kapital), atau 'INDONESIA'
    filtered_df = filtered_df[
        ~filtered_df["Kabupaten/Kota"].isin(["nan", "Catatan", "INDONESIA", ""]) &
        ~filtered_df["Kabupaten/Kota"].str.isupper() &
        ~filtered_df["Kabupaten/Kota"].str.contains(r"Provinsi|Indonesia|Jumlah|Total|Catatan", case=False, na=False)
    ].copy()
    
    # 3. Cleaning Nilai (replace '-' dan whitespace -> 0, cast to float)
    num_df = filtered_df.iloc[:, 1:].copy()
    num_df = num_df.map(lambda x: 0.0 if str(x).strip() in ["-", ""] else x)
    num_df = num_df.apply(pd.to_numeric, errors="coerce").fillna(0.0).astype(float)
    num_df.columns = year_labels
    
    # 4. Agregasi Dinamis (Jika ada subkomoditas / nama tahun berulang)
    has_duplicate_years = len(year_labels) > len(set(year_labels))
    if is_komoditas and has_duplicate_years:
        # Vektorisasi: Transpose -> Groupby level=0 -> sum -> Transpose kembali
        agg_num = num_df.T.groupby(level=0, sort=False).sum().T
    else:
        agg_num = num_df.copy()
        
    # Pastikan urutan tahun konsisten: 2018 s.d. 2024
    agg_num = agg_num[TARGET_YEARS]
    
    # 5. Rename Kolom dengan Prefix Nama File
    agg_num.columns = [f"{base_name}_{y}" for y in TARGET_YEARS]
    
    # Gabungkan kolom wilayah dan nilai hasil olahan
    clean_df = pd.concat([
        filtered_df[["Kabupaten/Kota"]].reset_index(drop=True),
        agg_num.reset_index(drop=True)
    ], axis=1)
    
    return clean_df

### Cell 4: Iterasi, Merge, dan Cleaning Wilayah (511 Kab/Kota)
Memproses seluruh 14 file, melakukan penggabungan berurutan (*outer merge*) berdasarkan kolom `Kabupaten/Kota`, dan memfilter tepat **511 Kabupaten/Kota** sesuai standar spasial GeoJSON (menghilangkan 3 Daerah Otonomi Baru pemekaran pasca-511).

In [ ]:
# Daftar 3 file indikator tambahan
indikator_files = [
    "Garis Kemiskinan.xlsx",
    "Indeks Kedalaman Kemiskinan (P1).xlsx",
    "Prevalensi Ketidakcukupan Konsumsi Pangan.xlsx"
]

all_dfs = []

print("[*] Memproses 11 File Komoditas...")
for fname in komoditas_files:
    fpath = os.path.join(DATA_DIR, fname)
    if os.path.exists(fpath):
        df_proc = process_excel(fpath, is_komoditas=True)
        all_dfs.append(df_proc)
        print(f"  [V] {fname:<32} -> Dimensi: {df_proc.shape}")

print("\n[*] Memproses 3 File Indikator Tambahan...")
for fname in indikator_files:
    fpath = os.path.join(DATA_DIR, fname)
    if os.path.exists(fpath):
        df_proc = process_excel(fpath, is_komoditas=False)
        all_dfs.append(df_proc)
        print(f"  [V] {fname:<32} -> Dimensi: {df_proc.shape}")

# Outer merge secara berurutan pada kolom 'Kabupaten/Kota'
print(f"\n[*] Menggabungkan {len(all_dfs)} DataFrame secara Outer Merge...")
df_master = reduce(lambda left, right: pd.merge(left, right, on="Kabupaten/Kota", how="outer"), all_dfs)
print(f"Dimensi sebelum filter 511 Kab/Kota : {df_master.shape}")

# Filter 511 Kabupaten/Kota:
# Data BPS memuat 514 Kab/Kota. Tiga DOB pasca pemekaran peta 511 adalah:
# 'Muna Barat', 'Buton Tengah', dan 'Buton Selatan'.
dob_exclude = ["Muna Barat", "Buton Tengah", "Buton Selatan"]
df_master = df_master[~df_master["Kabupaten/Kota"].isin(dob_exclude)].reset_index(drop=True)

print(f"Dimensi final (target 511 Kab/Kota) : {df_master.shape}")
print(f"Total Missing Value (NaN)          : {df_master.isna().sum().sum()}")

### Cell 5: Statistik Deskriptif & Export Master Data
Menampilkan ringkasan struktur DataFrame, statistik deskriptif untuk setiap komoditas dan indikator, serta mengekspor hasil akhirnya ke format CSV.

In [ ]:
print("=" * 80)
print("RINGKASAN STRUKTUR DATA (INFO)")
print("=" * 80)
df_master.info(verbose=False)

print("\n" + "=" * 80)
print("RINGKASAN STATISTIK DESKRIPTIF (SAMPLE 5 KOLOM INDIKATOR PERTAMA)")
print("=" * 80)
print(df_master.iloc[:, 1:6].describe())

# Export ke CSV tanpa index
output_csv = "master_data_pengeluaran_kemiskinan_2018_2024.csv"
df_master.to_csv(output_csv, index=False)
print(f"\n[V] Berhasil diekspor ke: {output_csv}")
print(f"Total Baris: {len(df_master)} (511 Kab/Kota) | Total Kolom: {len(df_master.columns)} (1 Wilayah + 14 Indikator x 7 Tahun)")

# Tampilkan preview data teratas
df_master.head()

Cell 6: Fungsi Pemrosesan Subkomoditas ke Format Long

In [ ]:
def process_subkomoditas_long(file_path, target_years):
    """
    Mengekstrak data rincian subkomoditas dengan membaca MultiIndex 3 level header Excel:
    - Level 0: Judul Tabel
    - Level 1: Komoditas Utama (KAPITAL) & Subkomoditas
    - Level 2: Tahun
    Melakukan unpivot / melt ke dalam format long 5 kolom.
    """
    # 1. Load Excel dengan MultiIndex header 3 level
    df = pd.read_excel(file_path, header=[0, 1, 2])
    
    # Kolom Wilayah berada di kolom pertama (index 0)
    region_series = df.iloc[:, 0].astype(str).str.strip()
    region_series = region_series.replace({"Pangkajene Dan Kepulauan": "Pangkajene dan Kepulauan"})
    
    # Filter 511 Kabupaten/Kota (membuang provinsi, catatan, dan 3 DOB pemekaran)
    mask = (
        ~region_series.isin(["nan", "Catatan", "INDONESIA", ""]) &
        ~region_series.str.isupper() &
        ~region_series.str.contains(r"Provinsi|Indonesia|Jumlah|Total|Catatan", case=False, na=False) &
        ~region_series.isin(["Muna Barat", "Buton Tengah", "Buton Selatan"])
    )
    
    df_clean = df.loc[mask]
    kabupaten = region_series.loc[mask].values
    
    # Ambil Komoditas Utama dari header Level 1 (kolom pertama data)
    main_comm = str(df.columns[1][1]).strip().upper()
    
    # 2. Identifikasi Kolom Subkomoditas murni (drop kolom total agregat komoditas utama)
    data_cols = []
    sub_names = []
    years = []
    
    for col in df.columns[1:]:
        lvl0, sub, year_raw = col
        sub_clean = str(sub).strip()
        
        # PENTING: Lewati kolom total kelompok komoditas utama agar tidak double-count pada Treemap/Sunburst
        if sub_clean.upper() == main_comm:
            continue
            
        try:
            y = int(float(str(year_raw).split(".")[0]))
            if y in target_years:
                data_cols.append(col)
                sub_names.append(sub_clean.lower())
                years.append(y)
        except (ValueError, TypeError):
            continue
            
    # 3. Cleaning Nilai (replace '-' dan whitespace -> 0, cast ke float)
    num_df = df_clean[data_cols].map(lambda x: 0.0 if str(x).strip() in ["-", ""] else x)
    num_df = num_df.apply(pd.to_numeric, errors="coerce").fillna(0.0).astype(float)
    
    # Berikan nama kolom gabungan unik untuk keperluan unpivot/melt
    num_df.columns = [f"{s}___{y}" for s, y in zip(sub_names, years)]
    num_df.insert(0, "Kabupaten/Kota", kabupaten)
    
    # 4. Unpivot / Melt menjadi Long Format
    melted = num_df.melt(id_vars=["Kabupaten/Kota"], var_name="sub_year", value_name="Pengeluaran")
    melted[["Subkomoditas", "Tahun"]] = melted["sub_year"].str.split("___", expand=True)
    melted["Tahun"] = melted["Tahun"].astype(int)
    melted["Komoditas_Utama"] = main_comm
    
    # Susun persis 5 kolom target sesuai requirements
    melted = melted[["Kabupaten/Kota", "Komoditas_Utama", "Subkomoditas", "Tahun", "Pengeluaran"]]
    return melted

print("Fungsi process_subkomoditas_long berhasil didefinisikan.")


Cell 7: Iterasi 11 File Komoditas, Concat, & Export

In [ ]:
target_years_int = [y for y in range(2018, 2025)]  # 2018 - 2024
all_long_dfs = []

print("=" * 75)
print("MEMPROSES 11 FILE KOMODITAS KE LONG / TIDY FORMAT")
print("=" * 75)

for fname in komoditas_files:
    fpath = os.path.join(DATA_DIR, fname)
    if os.path.exists(fpath):
        df_long = process_subkomoditas_long(fpath, target_years_int)
        all_long_dfs.append(df_long)
        n_sub = df_long["Subkomoditas"].nunique()
        print(f"  [V] {fname:<30} -> {len(df_long):>6} baris | {n_sub:>2} Subkomoditas")

# 5. Gabungkan (Concat) seluruh dataframe komoditas
print("\n[*] Menggabungkan seluruh data long (pd.concat)... ")
df_subkomoditas_master = pd.concat(all_long_dfs, ignore_index=True)

# 6. Export ke CSV
out_long_csv = "master_subkomoditas_long.csv"
df_subkomoditas_master.to_csv(out_long_csv, index=False)

print("=" * 75)
print(f"[V] SUKSES! Master data subkomoditas diekspor ke: {out_long_csv}")
print(f"Total Baris               : {len(df_subkomoditas_master):,}")
print(f"Jumlah Kabupaten/Kota     : {df_subkomoditas_master['Kabupaten/Kota'].nunique()} (Tepat 511 Kab/Kota)")
print(f"Jumlah Komoditas Utama    : {df_subkomoditas_master['Komoditas_Utama'].nunique()}")
print(f"Jumlah Subkomoditas Unik  : {df_subkomoditas_master['Subkomoditas'].nunique()}")
print(f"Rentang Tahun             : {df_subkomoditas_master['Tahun'].min()} s.d. {df_subkomoditas_master['Tahun'].max()}")
print(f"Jumlah Nilai Null (NaN)   : {df_subkomoditas_master.isna().sum().sum()}")
print("=" * 75)

# Preview hasil
df_subkomoditas_master.head(10)


Cell 8: Contoh Penggunaan Visualisasi Hierarki Plotly Express

In [ ]:
# CONTOH PENGGUNAAN PLOTLY EXPRESS UNTUK VISUALISASI HIERARKI (UAS VISDAT)
import plotly.express as px

# 1. Filter contoh: Satu Kabupaten/Kota (misal Simeulue) pada Tahun 2024
df_sample = df_subkomoditas_master[
    (df_subkomoditas_master["Kabupaten/Kota"] == "Simeulue") & 
    (df_subkomoditas_master["Tahun"] == 2024)
]

# 2. Buat Treemap Hierarki (Level 1: Komoditas Utama -> Level 2: Subkomoditas)
fig_treemap = px.treemap(
    df_sample,
    path=["Komoditas_Utama", "Subkomoditas"],
    values="Pengeluaran",
    color="Pengeluaran",
    color_continuous_scale="RdBu",
    title="Hierarki Pengeluaran per Komoditas & Subkomoditas di Kab. Simeulue (2024)"
)
fig_treemap.update_layout(margin=dict(t=50, l=10, r=10, b=10))
# fig_treemap.show()

# 3. Buat Sunburst Chart Alternatif
fig_sunburst = px.sunburst(
    df_sample,
    path=["Komoditas_Utama", "Subkomoditas"],
    values="Pengeluaran",
    color="Komoditas_Utama",
    title="Sunburst Chart Pengeluaran Subkomoditas (2024)"
)
fig_sunburst.update_layout(margin=dict(t=50, l=10, r=10, b=10))
# fig_sunburst.show()

print("[V] Visualisasi Treemap dan Sunburst siap digunakan langsung dari dataset ini!")
